# Sudoku solving with PySCIPOpt

Example derived from [PySCIPOpt repository](https://github.com/scipopt/PySCIPOpt) under MIT license.

### Input data

Some input data to get started (a valid soudoku).
This code snippet will diplay a Sudoku grid that can be edited to change the problem (the value of puzzle is automatically changed).

In [ ]:
import sudoku_utils as utils

# Example Sudoku values
X = None
default_grid = [
    5, 3, X, X, 7, X, X, X, X,
    6, X, X, 1, 9, 5, X, X, X,
    X, 9, 8, X, X, X, X, 6, X,
    8, X, X, X, 6, X, X, X, 3,
    4, X, X, 8, X, 3, X, X, 1,
    7, X, X, X, 2, X, X, X, 6,
    X, 6, X, X, X, X, 2, 8, X,
    X, X, X, 4, 1, 9, X, X, 5,
    X, X, X, X, 8, X, X, 7, 9,
]

puzzle = utils.input_sudoku(default_grid)
puzzle

### Modeling

Creating an binary integer linear optimization problem.

In [ ]:
import pyscipopt as scip


Variables = dict[tuple[int, int, int], scip.Variable]


def build_model(grid: list[int | None]) -> tuple[scip.Model, Variables]:    
    assert len(grid) == 9 * 9

    m = scip.Model()

    # Create a binary variable for every field and value
    x = {}
    for i in range(9):
        for j in range(9):
            for k in range(9):
                name = str(i) + ',' + str(j) + ',' + str(k)
                x[i, j, k] = m.addVar(name, vtype='B')
    
    # Fill in initial values
    for i in range(9):
        for j in range(9):
            if grid[j + 9 * i] != None:
                m.addCons(x[i, j, grid[j + 9 * i] - 1] == 1)
    
    # Only one digit in every field
    for i in range(9):
        for j in range(9):
            m.addCons(scip.quicksum(x[i, j, k] for k in range(9)) == 1)
    
    # Set up row and column constraints
    for ind in range(9):
        for k in range(9):
            m.addCons(scip.quicksum(x[ind, j, k] for j in range(9)) == 1)
            m.addCons(scip.quicksum(x[i, ind, k] for i in range(9)) == 1)
    
    # Set up square constraints
    for row in range(3):
        for col in range(3):
            for k in range(9):
                lhs = scip.quicksum(
                    x[i + 3 * row, j + 3 * col, k]
                    for i in range(3)
                    for j in range(3)
                )
                m.addCons(lhs == 1)

    return m, x


def get_solution(m: scip.Model, x: Variables) -> list[int] | None:
    """Pull the solution as a flat list of 81 digits, or None if infeasible."""
    if m.getStatus() != 'optimal':
        return None
    sol = [0] * (9 * 9)
    for i in range(9):
        for j in range(9):
            for k in range(9):
                if m.getVal(x[i, j, k]) == 1:
                    sol[j + 9 * i] = k + 1
    return sol

### Solution

The solver computes a solution (if it exists), and we display it in the notebook.

In [ ]:
model, variables = build_model(puzzle.value)

# Solve
model.hideOutput()
model.optimize()

solution = get_solution(model, variables)
utils.print_sudoku(solution, given=puzzle.value)